In [22]:
from ingest import load_faq_data, build_index
documents = load_faq_data()

documents_llm = []

for doc in documents:
    if doc["course"] == "llm-zoomcamp":
        documents_llm.append(doc)

documents = documents_llm
index = build_index(documents)

doc = documents[0]

from pydantic import BaseModel
class Questions(BaseModel):
    questions: list[str]

from concurrent.futures import ThreadPoolExecutor
from evaluation_utils import map_progress
import json
from evaluation_utils import llm_structured_retry

data_gen_instructions = """
You emulate a student who's taking our course.
Formulate 5 questions this student might ask based on a FAQ record. The record
should contain the answer to the questions, and the questions should be complete and not too short.
If possible, use as fewer words as possible from the record.

The output should resemble how people ask questions
on the internet. Not too formal, not too short, not too long.
""".strip()

import pandas as pd
df_ground_truth = pd.read_csv("data/ground_truth-new.csv")
ground_truth = df_ground_truth.to_dict(orient="records")

doc_idx = {}
for doc in documents:
    doc_idx[doc["id"]] = doc

df_answers = pd.read_csv("data/rag-answers-new.csv")
answers = df_answers.to_dict(orient="records")

from pydantic import Field
from typing import Literal

class AnswerEvaluation(BaseModel):
    reasoning: str = Field(
        description="Reasoning about the quality of the answer."
    )
    score: Literal["good", "bad"] = Field(
        description="'good' if the answer is correct and complete, 'bad' otherwise."
    )

aqa_judge_instructions = """
You are an expert evaluator. You will be given:
1. A question from a student
2. The original answer from the FAQ (ground truth)
3. An answer generated by an AI assistant

Your task is to decide if the AI answer is semantically equivalent to
the original answer.

Rules:
- The AI answer does NOT need to be word-for-word identical
- It should convey the same key information
- Extra detail is fine as long as the core answer is correct
- Mark 'bad' only if the AI answer is wrong or misses the key point

Be fair and focus on correctness, not style.
""".strip()

aqa_judge_prompt = """
Question:
{question}

Original Answer (ground truth):
{answer_orig}

AI Answer:
{answer_llm}
""".strip()

df_agent = pd.read_csv("data/agent-answers.csv")
agent_answers = df_agent.to_dict(orient="records")

class AgentEvaluation(BaseModel):
    answer_reasoning: str = Field(
        description="Reasoning about whether the final answer is correct."
    )
    answer_score: Literal["good", "bad"] = Field(
        description="'good' if the final answer matches the original answer."
    )
    trajectory_reasoning: str = Field(
        description="Reasoning about whether the tool calls were useful."
    )
    trajectory_score: Literal["good", "bad"] = Field(
        description="'good' if the tool calls were reasonable for the question."
    )

agent_judge_instructions = """
You are an expert evaluator. You will be given:
1. A question from a student
2. The original answer from the FAQ (ground truth)
3. An answer generated by an AI agent
4. The tool calls made by the agent

Evaluate two things:

Answer quality:
- Does the agent answer match the original answer?
- It does not need to be word-for-word identical.
- It should contain the same key information.

Trajectory quality:
- Were the search queries relevant to the question?
- Did the queries include important keywords from the question?
- Did the agent avoid duplicate or unnecessary tool calls?
- If it made multiple searches, did the later searches refine the query?
- Was the number of search calls reasonable? Usually 1 is enough, 2-3
  can be okay, and more than 3 needs a clear reason.
- Did the tool calls support the final answer?

Mark answer_score as 'good' if the final answer is correct.
Mark trajectory_score as 'good' if the tool calls were reasonable.
""".strip()

agent_judge_prompt = """
Question:
{question}

Original Answer (ground truth):
{answer_orig}

Agent Answer:
{answer_agent}

Tool Calls:
{tool_calls}
""".strip()

In [23]:

def text_search(query):
    boost_dict = {"question": 3.0, "section": 0.5}

    return index.search(
        query,
        num_results=5,
        boost_dict=boost_dict
    )

def compute_relevance_text(q):
    doc_id = q["document"]
    results = text_search(query=q["question"])

    relevance = []
    for d in results:
        relevance.append(int(d["id"] == doc_id))

    return relevance

from tqdm.auto import tqdm
def compute_relevance_total_text(ground_truth):
    relevance_total = []

    for q in tqdm(ground_truth):
        relevance = compute_relevance_text(q)
        relevance_total.append(relevance)

    return relevance_total

def compute_relevance(q, search_function):
    doc_id = q["document"]
    results = search_function(query=q["question"])

    relevance = []
    for d in results:
        relevance.append(int(d["id"] == doc_id))

    return relevance

def compute_relevance_total(ground_truth, search_function):
    relevance_total = []

    for q in tqdm(ground_truth):
        relevance = compute_relevance(q, search_function)
        relevance_total.append(relevance)

    return relevance_total

def hit_rate(relevance):
    cnt = 0

    for line in relevance:
        if 1 in line:
            cnt = cnt + 1

    return cnt / len(relevance)

def mrr(relevance):
    total_score = 0.0

    for line in relevance:
        for rank in range(len(line)):
            if line[rank] == 1:
                total_score = total_score + 1 / (rank + 1)
                break

    return total_score / len(relevance)

def evaluate(ground_truth, search_function):
    relevance_total = compute_relevance_total(ground_truth, search_function)

    return {
        "hit_rate": hit_rate(relevance_total),
        "mrr": mrr(relevance_total),
    }


def search_boosts(query, question_boost, answer_boost, section_boost):
    boost_dict = {
        "question": question_boost,
        "section": section_boost,
        "answer": answer_boost,
    }

    return index.search(
        query,
        num_results=5,
        boost_dict=boost_dict,
    )

In [24]:
from dotenv import load_dotenv
load_dotenv()

import os
from openai import OpenAI
openrouter_client = OpenAI(
    api_key=os.getenv("OPENROUTER_API_KEY"),
    base_url="https://openrouter.ai/api/v1"
)

In [ ]:
from evaluation_utils import calc_total_price

def evaluate_agent_answer(rec, model="gpt-5.4-mini"):
    tool_calls = rec["tool_calls"]

    if isinstance(tool_calls, str):
        tool_calls = json.loads(tool_calls)

    prompt = agent_judge_prompt.format(
        question=rec["question"],
        answer_orig=rec["answer_orig"],
        answer_agent=rec["answer_agent"],
        tool_calls=json.dumps(tool_calls, indent=2),
    )

    result, usage = llm_structured_retry(
        openrouter_client,
        agent_judge_instructions,
        prompt,
        AgentEvaluation,
        model=model,
    )

    return result, usage

def judge_agent_record(rec):
    agent_eval, usage = evaluate_agent_answer(rec, model="poolside/laguna-m.1:free")

    result = {
        "question": rec["question"],
        "document": rec["document"],
        "answer_score": agent_eval.answer_score,
        "answer_reasoning": agent_eval.answer_reasoning,
        "trajectory_score": agent_eval.trajectory_score,
        "trajectory_reasoning": agent_eval.trajectory_reasoning,
    }

    return result, usage

with ThreadPoolExecutor(max_workers=6) as pool:
    results = map_progress(pool, agent_answers, judge_agent_record)

agent_evaluations = []
usages = []
for evaluation, usage in results:
    agent_evaluations.append(evaluation)
    usages.append(usage)

df_agent_eval = pd.DataFrame(agent_evaluations)
df_agent_eval.to_csv("data/agent-evaluations.csv", index=False)

In [32]:
def search(query: str) -> list[dict]:
    """
    Search the FAQ database for entries matching the given query.
    """
    return index.search(
        query,
        num_results=5,
        boost_dict={"question": 1.0, "answer": 2.0, "section": 0.1},
        filter_dict={"course": "llm-zoomcamp"}
    )

from toyaikit.tools import Tools
from toyaikit.chat.runners import OpenAIResponsesRunner

agent_tools = Tools()
agent_tools.add_tool(search)

instructions = """
You're a course teaching assistant. Answer student questions based on
the FAQ search results. Use the search tool before answering.
""".strip()

from toyaikit.llm import OpenAIClient
runner = OpenAIResponsesRunner(
    tools=agent_tools,
    developer_prompt=instructions,
    llm_client=OpenAIClient(model="poolside/laguna-m.1:free", client = openrouter_client)
)

rec = ground_truth[0]

result = runner.loop(prompt=rec["question"])

def extract_tool_calls(messages):
    tool_calls = []

    for message in messages:
        if isinstance(message, dict):
            continue

        if message.type == "function_call":
            tool_calls.append({
                "name": message.name,
                "arguments": message.arguments,
            })

    return tool_calls

def generate_agent_answer(rec):
    doc_id = rec["document"]
    original_doc = doc_idx[doc_id]

    result = runner.loop(prompt=rec["question"])

    tool_calls = extract_tool_calls(result.all_messages)

    answer_record = {
        "question": rec["question"],
        "answer_agent": result.last_message,
        "answer_orig": original_doc["answer"],
        "tool_calls": tool_calls,
        "cost": result.cost.total_cost,
        "document": doc_id,
    }

    return answer_record

with ThreadPoolExecutor(max_workers=6) as pool:
    agent_answers = map_progress(pool, ground_truth[:50], generate_agent_answer)

df_agent = pd.DataFrame(agent_answers)
df_agent.to_csv("data/agent-answers.csv", index=False)

/media/bsh/0667a20f-7143-4a55-90cf-b3dc1ae94d49/bahar/Documents/Projects/python/llm-zoomcamp-homework/.venv/lib/python3.14/site-packages/toyaikit/chat/runners.py:283: UnknownModelWarning: No pricing data for model 'free'. Register it with PricingConfig.register_model(...) to get cost calculations.
  cost_info = self.pricing_config.calculate_cost(


[{'name': 'search', 'arguments': '{"query": "late join course"}'},
 {'name': 'search', 'arguments': '{"query": "join late registration"}'}]

In [27]:

def evaluate_aqa(question, answer_orig, answer_llm, model="gpt-5.4-mini"):
    prompt = aqa_judge_prompt.format(
        question=question,
        answer_orig=answer_orig,
        answer_llm=answer_llm
    )

    result, usage = llm_structured_retry(
        openrouter_client,
        aqa_judge_instructions,
        prompt,
        AnswerEvaluation,
        model=model,
    )

    return result, usage

def judge_record(rec):
    eval_result, usage = evaluate_aqa(
        question=rec["question"],
        answer_orig=rec["answer_orig"],
        answer_llm=rec["answer_llm"]
    )

    result = {
        "question": rec["question"],
        "document": rec["document"],
        "score": eval_result.score,
        "reasoning": eval_result.reasoning,
    }

    return result, usage


with ThreadPoolExecutor(max_workers=6) as pool:
    results = map_progress(pool, answers, judge_record)

evaluations = []
usages = []
for evaluation, usage in results:
    evaluations.append(evaluation)
    usages.append(usage)

df_eval = pd.DataFrame(evaluations)
df_eval.to_csv("data/rag-evaluations-new.csv", index=False)

good_count = (df_eval["score"] == "good").sum()
total_count = len(df_eval)
print(f"Good: {good_count}/{total_count} = {good_count/total_count:.2%}")
df_eval[df_eval["score"] == "bad"].head()


AnswerEvaluation(reasoning='The AI answer preserves the key meaning of the ground truth: late joining is allowed, and certificate eligibility depends on submitting the project while submissions are still open. This is semantically equivalent.', score='good')

In [21]:
from evaluation_utils import RAGWithUsage

assistant = RAGWithUsage(
    index=index,
    llm_client=openrouter_client,
)

def generate_rag_answer(rec):
    question = rec["question"]
    doc_id = rec["document"]
    original_doc = doc_idx[doc_id]

    answer_llm = assistant.rag(question)
    answer_orig = original_doc["answer"]

    result = {
        "question": question,
        "answer_llm": answer_llm,
        "answer_orig": answer_orig,
        "document": doc_id,
    }

    return result

with ThreadPoolExecutor(max_workers=6) as pool:
    results = map_progress(pool, ground_truth, generate_rag_answer)

answers = []
for answer_record in results:
    answers.append(answer_record)

df_answers = pd.DataFrame(answers)
df_answers.to_csv("data/rag-answers-new.csv", index=False)


{'question': 'Is it okay to join the course late if I just found it now?',
 'answer_llm': 'Yes, you can still join now. If you want to receive a certificate, make sure to submit your project while submissions are still being accepted.',
 'answer_orig': 'Yes, but if you want to receive a certificate, you need to submit your project while we’re still accepting submissions.',
 'document': '74eb249bbf'}

In [5]:

def generate_ground_truth(doc):
    user_prompt = json.dumps(doc)

    out, usage = llm_structured_retry(
        openrouter_client,
        data_gen_instructions,
        user_prompt,
        Questions,
        model="poolside/laguna-m.1:free"
    )

    results = []

    for q in out.questions:
        results.append({
            "question": q,
            "document": doc["id"]
        })

    return results, usage

from evaluation_utils import map_progress
with ThreadPoolExecutor(max_workers=6) as pool:
    results = map_progress(pool, documents, generate_ground_truth)

ground_truth = []
usages = []

for records, usage in results:
    ground_truth.extend(records)
    usages.append(usage)
    print(usage.input_tokens)
    print(usage.output_tokens)

from evaluation_utils import calc_price
total_cost = 0.0

for usage in usages:
    cost = calc_price(usage)
    total_cost = total_cost + cost["total_cost"]

total_cost

from evaluation_utils import calc_total_price
calc_total_price(usages)

import pandas as pd
df_ground_truth = pd.DataFrame(ground_truth)
df_ground_truth.to_csv("data/ground_truth-new.csv", index=False)

  0%|          | 0/103 [00:00<?, ?it/s]

KeyboardInterrupt: 